# TravelMind Coding Exercise 2: Intermediate
### Give after ~60% of Day 6 (parallelization, orchestrator-workers, evaluator-optimizer)

Traffic tripled after a fare sale, the complaint pile is growing, and Sofia in compliance wants every apology letter checked before it ships. You are handing part of the path to the model now. Cost goes up, so does flexibility. Measure both.

**What is inside:**
- Setup you run once (model, mock tools, a meter that also reads multi-agent runs).
- Ten tasks mixing MCQ, choose-an-option, fill-the-blank, a dials table, a flowchart to complete, spot-the-errors, debug-and-fix, predict-and-trace, and two implement-from-a-flow builds.

**Rules:** anchor booking PNR `JX48Q2`, surname `Rao`, Gold, `BLR-DEL` cancelled. Run setup first. Placeholders are `____` or `# TODO`.

## How to run
- **VS Code:** Python 3.10+ kernel, boto3 with Bedrock access to Haiku 4.5 in `us-east-1`, run top to bottom.
- **Colab:** upload, run install, set AWS env vars and `AWS_REGION`, run top to bottom.

In [ ]:
%pip install -q strands-agents strands-agents-tools nest_asyncio matplotlib

In [ ]:
import os, time, json, asyncio, contextlib
import nest_asyncio
nest_asyncio.apply()

from strands import Agent, tool
from strands.models import BedrockModel

REGION   = os.environ.get("AWS_REGION", "us-east-1")
HAIKU_ID = "us.anthropic.claude-haiku-4-5-20251001-v1:0"
haiku    = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0.3)
haiku_t0 = BedrockModel(model_id=HAIKU_ID, region_name=REGION, temperature=0)   # steady critics
print("Models ready.")

In [ ]:
# Mock airline operations (same as Notebook 1).
_PNRS = {"JX48Q2": {"surname": "Rao", "passenger_id": "P-100294", "loyalty_tier": "Gold",
    "segments": [{"flight": "6E-317", "from": "BLR", "to": "DEL", "dep": "2026-07-05T08:10",
                  "status": "CANCELLED", "fare_basis": "TA21R"},
                 {"flight": "6E-512", "from": "DEL", "to": "BOM", "dep": "2026-07-05T13:40",
                  "status": "ON TIME", "fare_basis": "TA21R"}],
    "ancillaries": {"seat": "14C (paid)", "bags": 1}}}
_FARE_RULES = {"TA21R": {"refundable": False, "change_fee": 3000, "currency": "INR",
    "notes": "Non-refundable on voluntary change. Airline-caused (involuntary) changes waive fees."}}


@tool
def get_pnr(record_locator: str, surname: str) -> str:
    """Look up a booking by record locator and surname.

    Args:
        record_locator: 6-character PNR.
        surname: Passenger surname.
    Returns:
        JSON string of the booking, or an error.
    """
    rec = _PNRS.get(record_locator.upper())
    if not rec or rec["surname"].lower() != surname.lower():
        return json.dumps({"error": "PNR not found or surname mismatch"})
    return json.dumps(rec)


@tool
def get_fare_rules(fare_basis: str) -> str:
    """Return fare rules for a fare basis code.

    Args:
        fare_basis: Fare basis code, e.g. 'TA21R'.
    Returns:
        JSON string of fare rules.
    """
    return json.dumps(_FARE_RULES.get(fare_basis.upper(), {"error": "unknown fare basis"}))


@tool
def search_reaccommodation(origin: str, destination: str, after: str) -> str:
    """Find alternative flights after a disruption.

    Args:
        origin: Origin airport code.
        destination: Destination airport code.
        after: ISO datetime lower bound.
    Returns:
        JSON string list of candidate flights.
    """
    return json.dumps([{"flight": "6E-333", "from": origin, "to": destination, "dep": "2026-07-05T11:20", "seats": 4},
                       {"flight": "AI-809", "from": origin, "to": destination, "dep": "2026-07-05T15:05", "seats": 9}])


@tool
def get_loyalty(passenger_id: str) -> str:
    """Return loyalty tier and benefits.

    Args:
        passenger_id: Internal passenger id.
    Returns:
        JSON string of tier and benefits.
    """
    return json.dumps({"passenger_id": passenger_id, "tier": "Gold",
                       "benefits": ["priority rebooking", "waived change fee on same-day", "lounge access"]})


@tool
def check_refund_eligibility(record_locator: str, reason: str) -> str:
    """Assess refund eligibility from fare rules and reason.

    Args:
        record_locator: PNR.
        reason: Free-text reason.
    Returns:
        JSON string with eligibility and the controlling rule.
    """
    rec = _PNRS.get(record_locator.upper(), {})
    if not rec:
        return json.dumps({"error": "PNR not found"})
    rules = _FARE_RULES.get(rec["segments"][0]["fare_basis"], {})
    airline_caused = any(k in reason.lower() for k in ["cancel", "delay", "airline", "irrops"])
    return json.dumps({"eligible": airline_caused or rules.get("refundable", False),
                       "airline_caused": airline_caused, "rule": rules.get("notes", "")})

print("Tools ready.")

In [ ]:
# Meter: reads single-agent and multi-agent (Graph) token usage.
PRICES = {"haiku": (1.00, 5.00)}   # illustrative; verify before quoting
_LEDGER, RESULTS = [], {}

def usage_of(result):
    u = getattr(getattr(result, "metrics", None), "accumulated_usage", None)
    if u is None:
        u = getattr(result, "accumulated_usage", None) or {}
    return {"input": u.get("inputTokens", 0) or u.get("input_tokens", 0),
            "output": u.get("outputTokens", 0) or u.get("output_tokens", 0)}

def _nodes_run(result):
    seq = getattr(result, "execution_order", None) or getattr(result, "node_history", None) or []
    return [getattr(n, "node_id", None) or getattr(n, "id", None) or str(n) for n in seq]

def final_text(result):
    ids = _nodes_run(result)
    if ids:
        try:
            return str(result.results[ids[-1]].result)
        except Exception:
            pass
    return str(result)

def _record(res, tier="haiku"):
    _LEDGER.append((tier, usage_of(res), 1))

def metered(agent, prompt, tier="haiku"):
    res = agent(prompt); _record(res, tier); return str(res)

def metered_multi(orchestrator, prompt, tier="haiku"):
    res = orchestrator(prompt)
    _LEDGER.append((tier, usage_of(res), max(1, len(_nodes_run(res)))))
    return final_text(res), res

@contextlib.contextmanager
def meter(label):
    _LEDGER.clear(); t0 = time.time()
    try:
        yield
    finally:
        lat = time.time() - t0; cost = tin = tout = n = 0
        for tier, u, c in _LEDGER:
            pin, pout = PRICES[tier]
            cost += u["input"]/1e6*pin + u["output"]/1e6*pout; tin += u["input"]; tout += u["output"]; n += c
        RESULTS[label] = {"latency_s": round(lat, 2), "tokens": tin + tout, "calls": n, "cost_usd": round(cost, 6)}
        print(f"[{label}] calls/nodes={n} latency={round(lat,2)}s tokens={tin+tout} cost=${round(cost,6):.6f}")

print("Meter ready.")

## Task 1: MCQ, match the problem to the pattern

- **P-A:** "Cancelled my BLR-DEL, I'm Gold, refund OR next flight, and I paid for a seat, what happens to it?" One message, tangled asks, no clean category.
- **P-B:** every apology letter must pass a written policy and tone bar before sending.

Set your answers.

In [ ]:
# a) routing   b) orchestrator-workers   c) evaluator-optimizer   d) chaining
ANSWER_PA = "?"   # P-A: messy multi-part
ANSWER_PB = "?"   # P-B: quality gate before send
print(ANSWER_PA, ANSWER_PB)

## Task 2: Fill the blank, wire the orchestrator

When you pass an agent into `tools=[...]`, it becomes a callable tool named by its `name`. Complete the orchestrator so the model can delegate to all three specialists.

In [ ]:
flight_specialist = Agent(model=haiku, name="flight_specialist", system_prompt="Report flight status for a PNR. Verify identity.", tools=[get_pnr])
fare_specialist   = Agent(model=haiku, name="fare_specialist",   system_prompt="Explain fare rules and whether a change is involuntary (fees waived).", tools=[get_pnr, get_fare_rules])
refund_specialist = Agent(model=haiku, name="refund_specialist", system_prompt="Decide refund eligibility; never promise without checking.", tools=[get_pnr, check_refund_eligibility])

orchestrator = Agent(
    model=haiku, name="orchestrator",
    system_prompt="Read the message, consult only the specialists needed, then synthesize one reply.",
    tools=[____, ____, ____],     # TODO: the three specialists
)
print("orchestrator tools:", len(orchestrator.tool_names) if hasattr(orchestrator, "tool_names") else "wired")

## Task 3: Debug, one specialist is unreachable

The orchestrator below cannot reliably address one specialist. Run it, find the specialist the model cannot name, and fix it.

In [ ]:
flight = Agent(model=haiku, name="flight_specialist", system_prompt="Report flight status.", tools=[get_pnr])
fare   = Agent(model=haiku, name="fare_specialist",   system_prompt="Explain fare rules and waivers.", tools=[get_pnr, get_fare_rules])
refund = Agent(model=haiku, system_prompt="Decide refund eligibility.", tools=[get_pnr, check_refund_eligibility])   # something is off here

orch = Agent(model=haiku, name="orch",
             system_prompt="Delegate to specialists as needed, then synthesize one reply.",
             tools=[flight, fare, refund])

with meter("t3_debug"):
    print(metered(orch, "Rao, JX48Q2. Airline cancelled my flight. Am I owed a refund?"))

## Task 4: Spot the errors in the cost helper

This should total the cost of every call in a ledger. It has **two** bugs. Identify both.

```python
def block_cost(ledger):
    total = 0
    for tier, usage in ledger:
        p_out, p_in = PRICES[tier]                                  # line A
        total += usage["input"] * p_in + usage["output"] * p_out    # line B
    return total
```

Record your findings, then write the corrected two lines as a string.

In [ ]:
BUG_A = "____"        # TODO: what is wrong on line A
BUG_B = "____"        # TODO: what is wrong on line B
FIXED_LINES = "____"  # TODO: the corrected line A and line B as one string
print(BUG_A); print(BUG_B); print(FIXED_LINES)

## Task 5: Implement from the flow, parallel sectioning

Three checks are independent, so run them at the same time and merge. Wall-clock drops to the slowest one instead of the sum. Parallel buys time, not money.

```mermaid
flowchart TD
    In([Change request]) --> C[coordinator]
    C --> W1[fare_agent]
    C --> W2[reaccom_agent]
    C --> W3[loyalty_agent]
    W1 --> AGG[aggregator]
    W2 --> AGG
    W3 --> AGG
    AGG --> Out([One reply])
```

Implement `gather_change`: run the three specialists concurrently with `asyncio.gather` and `asyncio.to_thread(metered, agent, msg)`, then feed all three into the aggregator.

In [ ]:
fare_agent    = Agent(model=haiku, name="fare_agent",    system_prompt="Report only the fare rules relevant to a change.", tools=[get_pnr, get_fare_rules])
reaccom_agent = Agent(model=haiku, name="reaccom_agent", system_prompt="Report only re-accommodation options.", tools=[get_pnr, search_reaccommodation])
loyalty_agent = Agent(model=haiku, name="loyalty_agent", system_prompt="Report only loyalty benefits for a change.", tools=[get_pnr, get_loyalty])
aggregator    = Agent(model=haiku, name="aggregator",    system_prompt="Merge fare, re-accommodation, and loyalty findings into one clear reply.")

q = "Rao, JX48Q2. Airline cancelled BLR-DEL. Options to reach BOM today, and any Gold benefits?"

async def gather_change(msg):
    fare, reaccom, loyalty = ____     # TODO: gather the three concurrently (asyncio.gather + asyncio.to_thread)
    return metered(aggregator, "Fare:\n" + fare + "\n\nReaccom:\n" + reaccom + "\n\nLoyalty:\n" + loyalty)

with meter("t5_parallel"):
    print(asyncio.run(gather_change(q)))

## Task 6: Predict, then trace

Send Nadia's messy message to the Task 2 orchestrator. Before running, predict which specialists the model will call and for which part of her message.

> "Rao, JX48Q2. Cancelled BLR-DEL, I'm Gold, refund or next flight, and I paid for a seat, what about it?"

In [ ]:
# Predict first: list the specialist names you expect the orchestrator to call.
PREDICTED_CALLS = ["____", "____"]   # TODO: fill with expected specialist names

# Now run and compare.
messy = "Rao, JX48Q2. Cancelled BLR-DEL, I'm Gold, refund or next flight, and I paid for seat 14C, what about it?"
with meter("t6_orchestrator"):
    print(metered(orchestrator, messy))
print("I predicted:", PREDICTED_CALLS)

## Task 7: Fill the dials table

Versus a single-agent baseline, mark each dial **lower**, **higher**, or **same**.

| Pattern | Cost | Latency | Quality on a hard task |
|---|---|---|---|
| Routing | ____ | ____ | ____ |
| Parallel sectioning | ____ | ____ | ____ |
| Parallel voting | ____ | ____ | ____ |
| Orchestrator-workers | ____ | ____ | ____ |
| Evaluator-optimizer | ____ | ____ | ____ |

Record the parallel-sectioning row as a check on your intuition.

In [ ]:
SECTIONING_COST    = "____"   # TODO: lower / higher / same
SECTIONING_LATENCY = "____"   # TODO
SECTIONING_QUALITY = "____"   # TODO
print("sectioning:", SECTIONING_COST, SECTIONING_LATENCY, SECTIONING_QUALITY)

## Task 8: Complete the flowchart, then build the evaluator loop

Sofia's quality gate is a cyclic graph. Fill the two edge labels, then finish the build: the missing condition, the loop cap, and the state reset.

```mermaid
flowchart TD
    In([Draft]) --> D[draft]
    D --> C{critic}
    C -->|... fill 1| D
    C -->|... fill 2| P[publish]
```

The critic ends with exactly `APPROVED` or `REVISION NEEDED: <fixes>`, and the edges branch on those tokens.

In [ ]:
from strands.multiagent import GraphBuilder

GEN    = "You are TravelMind. Draft a warm, correct customer reply. Verify identity, use tools, never invent policy."
POLICY = ("Identity verified before sharing details; no refund promised without checking; involuntary changes waive "
          "fees (say so); cancellation gets duty-of-care language; warm and concise.")
CRIT   = POLICY + " Review the draft, then end with EXACTLY one line: 'APPROVED' or 'REVISION NEEDED: <fixes>'."

draft   = Agent(model=haiku,    name="draft",   system_prompt=GEN, tools=[get_pnr, get_fare_rules, search_reaccommodation])
critic  = Agent(model=haiku_t0, name="critic",  system_prompt=CRIT)
publish = Agent(model=haiku,    name="publish", system_prompt="Format the approved reply. Do not change its meaning.")

def needs_revision(state):
    r = state.results.get("critic")
    return bool(r) and "revision needed" in str(r.result).lower()

def is_approved(state):
    r = state.results.get("critic")
    if not r:
        return False
    return ____          # TODO: True only if approved AND not asking for revision

b = GraphBuilder()
b.add_node(draft, "draft"); b.add_node(critic, "critic"); b.add_node(publish, "publish")
b.add_edge("draft", "critic")
b.add_edge("critic", "draft",   condition=needs_revision)
b.add_edge("critic", "publish", condition=____)     # TODO: which condition sends it to publish?
b.set_entry_point("draft")
b.____                    # TODO: cap the loop (max node executions)
b.____                    # TODO: reset the draft state each pass
eval_graph = b.build()

q2 = "Rao, JX48Q2. Airline cancelled BLR-DEL. I want the change fee waived and the next flight to BOM."
with meter("t8_evaluator"):
    reply, res = metered_multi(eval_graph, q2)
    print("order:", _nodes_run(res)); print(reply)

## Task 9: Choose the design

A queue splits into exactly four fixed categories, one branch each, no overlap. Two designs:
- **Design 1:** routing with a cheap classifier.
- **Design 2:** an orchestrator that decides branches at runtime.

Pick one, and name what the loser wastes.

In [ ]:
CHOICE = "____"   # TODO: "Design 1" or "Design 2"
REASON = "____"   # TODO: one line, what the loser wastes
print(CHOICE, "-", REASON)

## Task 10: Fix the loop with no brakes

If the critic never approves, this graph loops until it runs up the bill. Add the two missing lines that make it safe.

In [ ]:
b2 = GraphBuilder()
b2.add_node(draft, "draft"); b2.add_node(critic, "critic"); b2.add_node(publish, "publish")
b2.add_edge("draft", "critic")
b2.add_edge("critic", "draft",   condition=needs_revision)
b2.add_edge("critic", "publish", condition=is_approved)
b2.set_entry_point("draft")
# TODO: add the two lines that cap the loop and reset draft state each pass

safe_graph = b2.build()
print("built:", safe_graph is not None)

## Done
You measured parallelization (time, not money), let the model delegate with orchestrator-workers, and capped a critic loop for quality. The path is now partly the model's. The last notebook goes to the extremes: peers with no boss, and a deterministic auditable machine.